# MA: M365 Master Calculus — Full Verification

**Source:** [M365_MASTER_CALCULUS.md](../docs/contracts/caio-m365/M365_MASTER_CALCULUS.md)

Every response satisfies:
- **Eq. 1–2:** Postcondition (ok⇒result/shape, ¬ok⇒error) — INV-CAIO-M365-001, L1
- **Eq. 3:** Idempotency (same key+body ⇒ cached) — INV-CAIO-M365-002, L2
- **Eq. 4:** Auth (key required ⇒ missing/invalid ⇒ 401) — INV-CAIO-M365-003, L3
- **Eq. 5:** Audit (one record per request when enabled) — INV-M365-AUDIT-001, L4

This notebook runs all MA verification scripts and asserts every artifact passes.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while (
    not (repo_root / "scripts" / "ci" / "verify_caio_m365_contract.py").exists()
    and repo_root != repo_root.parent
):
    repo_root = repo_root.parent

scripts = [
    "verify_caio_m365_contract.py",
    "verify_m365_idempotency.py",
    "verify_m365_auth.py",
    "verify_m365_audit.py",
]
for name in scripts:
    p = repo_root / "scripts" / "ci" / name
    r = subprocess.run([sys.executable, str(p)], cwd=str(repo_root), capture_output=True, text=True)
    print(name, "exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr or r.stdout)
    assert r.returncode == 0, f"{name} failed"
print("All scripts passed.")

In [ ]:
gen = repo_root / "configs" / "generated"
artifacts = {
    "contract": gen / "caio_m365_contract_verification.json",
    "idempotency": gen / "m365_idempotency_verification.json",
    "auth": gen / "m365_auth_verification.json",
    "audit": gen / "m365_audit_verification.json",
}
summary = {}
for k, path in artifacts.items():
    with open(path) as f:
        a = json.load(f)
    if k == "contract":
        summary[k] = a.get("postcondition_pass") and a.get("response_schema_pass")
    elif k == "idempotency":
        summary[k] = a.get("idempotency_pass")
    elif k == "auth":
        summary[k] = a.get("auth_pass")
    else:
        summary[k] = a.get("audit_pass")
print("Summary:", json.dumps(summary, indent=2))
assert all(summary.values()), "One or more MA verifications failed"